# EEG-Auswertung – BrewVis InfoVis Studienarbeit

**Modul:** Informationsvisualisierung | Prof. Dr. Dieter Meiller  
**Gerät:** biosignalsplux (OpenSignals, 1000 Hz, 16-bit)  
**Kanäle:** CH1 = Frontal-EEG, CH2 = Okzipital-EEG  
**Datum:** 10.06.2026

---

## Ablauf
1. Konfiguration & Abhängigkeiten
2. Daten einlesen
3. Video-Synchronisation & Abdeckungsprüfung
4. Vorverarbeitung
5. Globale + lokale Skalierung
6. Frequenzanalyse (PSD)
7. Zeitverlauf & Epochenvergleich
8. Spektralanalyse
9. **Auffälligkeits-Erkennung** (lokal-kontrastbasiert)
10. **Persistente Auffälligkeits-Perioden**
11. Top-Ereignisse mit Video-Screenshots
12. Video-Overlay (vollständig, stabile Darstellung)
13. Dashboard & Erkenntnisse
14. Erweiterte Analysen (Aufmerksamkeits-Trend, Kopplung Sehen↔Denken, Statistik)
15. Ergebnis-Tabelle & PDF-Report

## Glossar – EEG-Begriffe einfach erklärt (zuerst lesen!)

Da EEG-Analyse viele Fachbegriffe nutzt, hier eine Übersetzung in Alltagssprache:

### Die Frequenzbänder (Gehirnwellen)
Das Gehirn erzeugt elektrische Schwingungen. Je nach Geschwindigkeit (Frequenz in Hz = Schwingungen pro Sekunde) bedeuten sie unterschiedliche mentale Zustände:

| Band | Frequenz | Bedeutung in einfachen Worten |
|------|----------|-------------------------------|
| **Delta** | 0.5–4 Hz | Sehr langsame Wellen. Normal im Tiefschlaf. Wach + viel Delta = oft Augenbewegungen oder Messartefakte. |
| **Theta** | 4–8 Hz | Tiefe Konzentration ODER Schläfrigkeit. Tritt auf bei mentaler Anstrengung (Kopfrechnen) oder wenn man müde wird. |
| **Alpha** | 8–13 Hz | "Entspannte Wachheit". Steigt wenn man die Augen schließt oder entspannt/abschweift. **Viel Alpha = Person ist gerade nicht stark fokussiert.** |
| **Beta** | 13–30 Hz | Aktives Denken, Konzentration, Aufmerksamkeit. **Viel Beta = Person ist geistig aktiv und fokussiert.** |
| **Gamma** | 30–45 Hz | Sehr schnelle Wellen, intensive Verarbeitung. Schwer sauber zu messen, oft mit Muskelaktivität vermischt. |

### Wichtige Kennzahlen
- **Engagement-Index (β/[α+θ])**: Eine Formel, die misst wie "eingebunden"/aufmerksam jemand ist. Hoch = konzentriert, niedrig = gelangweilt/passiv. (Nach Pope et al. 1995, Standard in der Forschung.)
- **Beta/Alpha-Verhältnis**: Einfacher Aktivierungswert. >1.5 = eher aktiv/fokussiert, <0.8 = eher entspannt/passiv.
- **PSD (Power Spectral Density)**: Zeigt, wie viel "Energie" in jeder Frequenz steckt. Wie ein Equalizer in der Musik-App.
- **Relative Bandleistung (%)**: Wie viel Prozent der gesamten Gehirnaktivität auf ein Band entfällt.

### Was ist ein "Artefakt"?
Eine Störung im Signal, die NICHT vom Gehirn kommt: Augenblinzeln, Muskelanspannung (Kiefer, Nacken), Bewegung, oder elektrisches Brummen (50 Hz Netzstrom). Diese filtern wir teilweise heraus.

### Die zwei Messpunkte (Elektroden)
- **Frontal (CH1)**: Stirn-Bereich. Gut für Konzentration, Aufmerksamkeit, kognitive Last.
- **Okzipital (CH2)**: Hinterkopf, über dem Sehzentrum. Gut für visuelle Verarbeitung (was die Augen sehen).

> **Wichtiger Hinweis zur Interpretation:** Diese Auswertung basiert auf **einer einzigen Person** mit nur **zwei Elektroden**. Echte EEG-Studien nutzen 32–64 Elektroden und viele Probanden. Die Ergebnisse hier sind daher **explorativ** – sie zeigen Tendenzen und sind gut für ein Studienprojekt, aber keine wissenschaftlich abgesicherten Fakten.

## 0. Abhängigkeiten & Konfiguration

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt, welch, spectrogram
from scipy.ndimage import uniform_filter1d
from scipy.stats import zscore
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import cv2
from IPython.display import display, Image as IPImage
import json, os, re
from datetime import datetime, timedelta
from pathlib import Path

try:
    import h5py; H5_AVAILABLE = True
except ImportError:
    H5_AVAILABLE = False

print('Abhängigkeiten geladen.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# KONFIGURATION – alle Parameter zentral hier anpassen
# ─────────────────────────────────────────────────────────────────────

EEG_TXT_FILE = 'data/opensignals_00078046f44e_2026-06-10_10-18-30.txt'

VIDEO_FILES = [
    'data/brauprozess-2026-06-10 10-19-28.mp4',
    'data/brauprozess-vr-2026-06-10 10-35-35.mp4',
]

OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/screenshots', exist_ok=True)

# EEG
SAMPLING_RATE = 1000
ZERO_OFFSET   = 32768

# Frequenzbänder (Hz)
FREQ_BANDS = {
    'Delta': (0.5,  4),
    'Theta': (4,    8),
    'Alpha': (8,   13),
    'Beta':  (13,  30),
    'Gamma': (30,  45),
}
# Okabe-Ito Palette (Okabe & Ito, 2008) – für alle gängigen Formen von
# Farbfehlsichtigkeit (Deuteranopie, Protanopie, Tritanopie) optimiert.
BAND_COLORS = {
    'Delta': '#0072B2', 'Theta': '#56B4E9', 'Alpha': '#E69F00',
    'Beta':  '#D55E00', 'Gamma': '#009E73', 'Other': '#999999',
}

# ── Rendering ─────────────────────────────────────────────────────────
# RENDER_SECONDS: Anzahl Sekunden die gerendert werden.
# None = komplettes Video, Zahl = nur die ersten N Sekunden
RENDER_SECONDS = None   # ← z.B. 60 für Vorschau, None für alles

# ── Anomalie-Erkennung ────────────────────────────────────────────────
N_TOP_ANOMALIES       = 9       # Wie viele Top-Ereignisse anzeigen
MIN_ANOMALY_GAP_S     = 12.0    # Mindestabstand zwischen Ereignissen (s)

# Lokale Kontrast-Parameter
LOCAL_SHORT_S         = 1.0     # Kurzfenster für lokale Aktivität (s)
LOCAL_LONG_S          = 12.0    # Referenzfenster für Umgebung (s)
CONTRAST_CHANGE_THRESH = 1.8    # Kontrastverhältnis ab dem eine Änderung zählt

# Persistente Perioden
PERSIST_MIN_DURATION_S = 5.0    # Mindestdauer einer persistenten Periode (s)
PERSIST_THRESHOLD      = 1.5    # Kontrastverhältnis-Schwelle für Persistenz

# ── Video-Overlay ─────────────────────────────────────────────────────
OVERLAY_WINDOW_S  = 5     # Sichtbares EEG-Zeitfenster (s)
OVERLAY_HEIGHT    = 215   # Pixel-Höhe des Panels
OVERLAY_ALPHA     = 0.86  # Transparenz des Panel-Hintergrunds
MIN_COVERAGE_PCT  = 80    # Mindest-EEG-Abdeckung für Videos (%)

# ── Band-Farbe im Overlay: Farbwechsel nur alle N Sekunden ───────────
COLOR_UPDATE_S    = 2.0   # Farbblock-Dauer (reduziert Flimmern)

print('Konfiguration geladen.')
print(f'  Rendering: {"vollständig" if RENDER_SECONDS is None else str(RENDER_SECONDS)+"s"}')

## 1. Daten einlesen

In [ ]:
def parse_opensignals_txt(filepath):
    metadata = {}
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('# {'):
                raw = json.loads(line.strip().lstrip('# '))
                metadata = raw[list(raw.keys())[0]]
                break
    fs      = metadata.get('sampling rate', SAMPLING_RATE)
    columns = metadata.get('column', ['nSeq', 'DI', 'CH1', 'CH2', 'CH3'])
    df = pd.read_csv(filepath, sep='\t', comment='#', header=None,
                     names=columns, index_col=False)
    parts    = metadata.get('date', '2026-6-10').split('-')
    date_str = f'{parts[0]}-{int(parts[1]):02d}-{int(parts[2]):02d}'
    time_str = metadata.get('time', '00:00:00.000')
    for fmt in ('%Y-%m-%d %H:%M:%S.%f', '%Y-%m-%d %H:%M:%S'):
        try:
            t0 = datetime.strptime(f'{date_str} {time_str}', fmt); break
        except ValueError:
            continue
    df['time_s']    = df['nSeq'] / fs
    df['timestamp'] = t0 + pd.to_timedelta(df['time_s'], unit='s')
    return df, metadata, fs, t0


df_raw, meta, FS, EEG_T0 = parse_opensignals_txt(EEG_TXT_FILE)
EEG_DURATION_S = len(df_raw) / FS
EEG_END        = EEG_T0 + timedelta(seconds=EEG_DURATION_S)

print(f'Aufnahmestart : {EEG_T0}')
print(f'Aufnahmeende  : {EEG_END}')
print(f'Dauer         : {EEG_DURATION_S:.1f} s  ({EEG_DURATION_S/60:.1f} min)')
print(f'Samples       : {len(df_raw):,}  |  {FS} Hz')
df_raw.head()

## 2. Synchronisation & Abdeckungsprüfung

In [ ]:
class VideoSync:
    def __init__(self, video_path, eeg_t0, eeg_duration_s, fs):
        self.video_path = video_path
        self.fs         = fs
        self.name       = Path(video_path).name
        cap = cv2.VideoCapture(video_path)
        self.fps          = cap.get(cv2.CAP_PROP_FPS)
        self.total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self.width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        self.video_duration_s = self.total_frames / self.fps
        m = re.search(r'(\d{4}-\d{1,2}-\d{1,2})[_ ](\d{2}-\d{2}-\d{2})',
                       Path(video_path).stem)
        if not m: raise ValueError(f'Kein Timestamp in {self.name!r}')
        self.video_t0 = datetime.strptime(
            f'{m.group(1)} {m.group(2).replace("-",":")}', '%Y-%m-%d %H:%M:%S')
        self.offset_s    = (self.video_t0 - eeg_t0).total_seconds()
        self.eeg_start_s = self.offset_s
        self.eeg_end_s   = self.offset_s + self.video_duration_s
        self.overlap_s   = max(0, min(self.eeg_end_s, eeg_duration_s)
                               - max(self.eeg_start_s, 0))
        self.coverage_pct = (self.overlap_s / self.video_duration_s * 100
                              if self.video_duration_s > 0 else 0)

    def eeg_sample_at_frame(self, frame_idx):
        return int((frame_idx / self.fps + self.offset_s) * self.fs)

    def eeg_window_at_frame(self, fi, df_eeg, window_s=OVERLAY_WINDOW_S):
        c = self.eeg_sample_at_frame(fi)
        h = int(window_s / 2 * self.fs)
        return df_eeg.iloc[max(0, c-h): min(len(df_eeg), c+h)]

    def screenshot_at_eeg_time(self, eeg_t, path):
        fi = max(0, min(int((eeg_t - self.offset_s) * self.fps),
                         self.total_frames - 1))
        cap = cv2.VideoCapture(self.video_path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read(); cap.release()
        if ret: cv2.imwrite(path, frame)
        return fi, ret


all_syncs = []
for vf in VIDEO_FILES:
    if not os.path.exists(vf):
        print(f'⚠ Nicht gefunden: {vf}'); continue
    s = VideoSync(vf, EEG_T0, EEG_DURATION_S, FS)
    all_syncs.append(s)

usable  = [s for s in all_syncs if s.coverage_pct >= MIN_COVERAGE_PCT]
SYNC    = usable[0] if usable else None

print('══ EEG-Abdeckungsprüfung ══')
print(f'EEG: {EEG_T0}  →  {EEG_END}  ({EEG_DURATION_S:.0f}s)\n')
for s in all_syncs:
    ok  = s.coverage_pct >= MIN_COVERAGE_PCT
    bar = '█' * int(s.coverage_pct / 2) + '░' * (50 - int(s.coverage_pct / 2))
    print(f'{"✓" if ok else "✗"}  {s.name}')
    print(f'   Offset: {s.offset_s:.1f}s | Dauer: {s.video_duration_s:.0f}s')
    print(f'   [{bar}] {s.coverage_pct:.0f}%')
    if not ok and s.offset_s > EEG_DURATION_S:
        print(f'   → Startet {s.offset_s - EEG_DURATION_S:.0f}s nach EEG-Ende – weggelassen')
    print()
print(f'→ {len(usable)} von {len(all_syncs)} Videos nutzbar.')
if not usable: raise RuntimeError('Kein Video abgedeckt!')

In [ ]:
# Zeitstrahl
fig, ax = plt.subplots(figsize=(14, 2.8))
ax.barh(0, EEG_DURATION_S, left=0, height=0.4, color='#1a1a2e', alpha=0.9)
ax.text(EEG_DURATION_S/2, 0, f'EEG ({EEG_DURATION_S:.0f}s)',
        ha='center', va='center', color='white', fontsize=9, fontweight='bold')
vc = ['#E85D04', '#1E88E5']
for i, s in enumerate(all_syncs):
    y = -(i+1) * 0.65
    a = 0.9 if s.coverage_pct >= MIN_COVERAGE_PCT else 0.3
    ax.barh(y, s.video_duration_s, left=s.offset_s, height=0.4,
            color=vc[i%2], alpha=a)
    ax.text(s.offset_s + s.video_duration_s/2, y,
            f'{s.name[:28]} ({s.coverage_pct:.0f}%)',
            ha='center', va='center', fontsize=7.5,
            color='white' if a > 0.5 else '#666')
ax.axvline(EEG_DURATION_S, color='red', lw=1.5, ls='--',
           label=f'EEG Ende ({EEG_DURATION_S:.0f}s)')
ax.set_xlim(-15, max(EEG_DURATION_S, max(s.eeg_end_s for s in all_syncs))+40)
ax.set_yticks([]); ax.set_xlabel('Zeit ab EEG-Start (s)')
ax.set_title('EEG-Abdeckung der Videos', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_zeitstrahl.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Vorverarbeitung

In [ ]:
def preprocess(raw, fs=1000, zero_offset=32768):
    c = raw.astype(np.float64) - zero_offset
    nyq = fs / 2
    filt = sosfiltfilt(butter(4, [0.5/nyq, 45/nyq], btype='band', output='sos'), c)
    filt = sosfiltfilt(butter(4, [49/nyq, 51/nyq], btype='bandstop', output='sos'), filt)
    return c, filt


df_raw['frontal_c'],   df_raw['frontal_f']   = preprocess(df_raw['CH1'].values, FS)
df_raw['okzipital_c'], df_raw['okzipital_f'] = preprocess(df_raw['CH2'].values, FS)

n = 5 * FS
t = df_raw['time_s'][:n]
fig, axes = plt.subplots(2, 2, figsize=(16, 6), sharex=True)
for row, (label, rc, rf) in enumerate([
    ('Frontal (CH1)',    'frontal_c',   'frontal_f'),
    ('Okzipital (CH2)', 'okzipital_c', 'okzipital_f'),
]):
    axes[row,0].plot(t, df_raw[rc][:n], color='#888', lw=0.6)
    axes[row,0].set_title(f'{label} – Roh (zero-centered)')
    axes[row,0].set_ylabel('Amplitude (a.u.)')
    axes[row,1].plot(t, df_raw[rf][:n], color=BAND_COLORS['Alpha'], lw=0.7)
    axes[row,1].set_title(f'{label} – Gefiltert (BP 0.5–45 Hz, Notch 50 Hz)')
for ax in axes[1]: ax.set_xlabel('Zeit (s)')
plt.suptitle('Vorverarbeitung – erste 5 Sekunden', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_preprocessing.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Lokale Normalisierung für das Video-Overlay

Das Kernproblem bei konstanter EEG-Amplitude: Globale Perzentile (1–99%) ergeben  
eine starre Skala, auf der der gesamte Verlauf gleich groß aussieht – weil die Amplitude  
tatsächlich gleichmäßig ist.

**Lösung: lokal-normalisierte Darstellung**  
Für jeden Frame wird das Signal relativ zu seiner lokalen Umgebung (rollender Median ± rollende MAD)  
skaliert. Damit zeigt die Y-Achse: *"Wie stark weicht das Signal gerade von seinem eigenen  
Ruhepegel der letzten Sekunden ab?"*  

- Y-Achse: ± 4 σ (lokale Standardabweichungseinheiten)
- Grüner Bereich: ± 1.5 σ (normaler Bereich)
- Gelber Bereich: 1.5–2.5 σ
- Roter Bereich: > 2.5 σ (auffälliger Ausschlag relativ zur Umgebung)

In [ ]:
def local_normalize(signal, fs, window_s=6.0):
    """
    Normalisiert das Signal lokal:
    - Subtrahiert rollenden Median (entfernt langsame Drifts)
    - Teilt durch rollende MAD (lokale Streuung)
    Ergebnis: Signal in lokalen Standardabweichungseinheiten.
    Werte > 2 = deutlich auffälliger Ausschlag relativ zur Umgebung.
    """
    n = int(window_s * fs)
    # Rollender Median via uniform_filter auf das Signal
    rolling_med = uniform_filter1d(signal, size=n)
    centered    = signal - rolling_med
    # Rollende MAD (Median Absolute Deviation) als robuste Streuungsschätzung
    rolling_mad = uniform_filter1d(np.abs(centered), size=n)
    # Skalierung: verhindere Division durch 0
    rolling_mad = np.maximum(rolling_mad, np.percentile(np.abs(centered), 10))
    return centered / rolling_mad


print('Berechne lokale Normalisierung (Referenzfenster 6s)...')
df_raw['frontal_norm'] = local_normalize(df_raw['frontal_f'].values, FS, window_s=6.0)

# Statistiken
p_check = np.percentile(np.abs(df_raw['frontal_norm'].values), [50, 75, 90, 95, 99])
print(f'  Abs-Wert Perzentile der normierten Werte:')
for pct, val in zip([50,75,90,95,99], p_check):
    print(f'    {pct}. Perzentil: {val:.2f} σ')
print(f'  → Display-Bereich ±4σ zeigt {(np.abs(df_raw["frontal_norm"]) < 4).mean()*100:.1f}% aller Samples')

## 5. Frequenzanalyse (PSD)

In [ ]:
def band_power(signal, fs):
    freqs, psd = welch(signal, fs=fs, nperseg=fs*2)
    total = np.trapezoid(psd, freqs)
    return freqs, psd, {b: np.trapezoid(psd[(freqs>=lo)&(freqs<=hi)],
                                         freqs[(freqs>=lo)&(freqs<=hi)])/total
                         for b, (lo,hi) in FREQ_BANDS.items()}


freqs_f, psd_f, bp_f = band_power(df_raw['frontal_f'].values,   FS)
freqs_o, psd_o, bp_o = band_power(df_raw['okzipital_f'].values, FS)

print(f'{"Band":<8} {"Frontal":>10} {"Okzipital":>12}')
print('-'*32)
for b in FREQ_BANDS:
    print(f'{b:<8} {bp_f[b]*100:>9.1f}% {bp_o[b]*100:>11.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (label, fq, ps, bp) in zip(axes, [
    ('Frontal (CH1)',   freqs_f, psd_f, bp_f),
    ('Okzipital (CH2)', freqs_o, psd_o, bp_o),
]):
    ax.semilogy(fq, ps, color='#333', lw=1.2)
    for band, (lo, hi) in FREQ_BANDS.items():
        m = (fq>=lo)&(fq<=hi)
        ax.fill_between(fq, ps, where=m, alpha=0.4, color=BAND_COLORS[band],
                         label=f'{band} {bp[band]*100:.1f}%')
    ax.set_xlim(0.5, 45); ax.set_xlabel('Frequenz (Hz)'); ax.set_ylabel('PSD (log)')
    ax.set_title(f'{label} – Leistungsdichtespektrum', fontweight='bold')
    ax.legend(fontsize=8)
plt.suptitle('EEG-Frequenzspektrum (Welch-PSD)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_psd.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Zeitverlauf & Epochenvergleich

In [ ]:
def rolling_bp(signal, fs, window_s=2.0, step_s=0.5):
    wn, sn = int(window_s*fs), int(step_s*fs)
    rows = []
    for start in range(0, len(signal)-wn, sn):
        chunk = signal[start:start+wn]
        fq, ps = welch(chunk, fs=fs, nperseg=min(wn,512))
        tot = np.trapezoid(ps, fq) + 1e-12
        row = {'time_s': (start+wn//2)/fs}
        for b,(lo,hi) in FREQ_BANDS.items():
            idx = (fq>=lo)&(fq<=hi)
            row[b] = np.trapezoid(ps[idx], fq[idx])/tot
        rows.append(row)
    return pd.DataFrame(rows)


print('Berechne rollende Bandleistung...')
df_roll_f = rolling_bp(df_raw['frontal_f'].values,   FS)
df_roll_o = rolling_bp(df_raw['okzipital_f'].values, FS)
df_roll_f['engagement'] = (df_roll_f['Beta'] /
    (df_roll_f['Alpha'] + df_roll_f['Theta'] + 1e-6))
print(f'  {len(df_roll_f):,} Zeitpunkte')

In [ ]:
# Epochen
epochs = {'Baseline': (0, SYNC.offset_s)}
for s in usable:
    epochs[Path(s.video_path).stem[:22]] = (
        max(s.eeg_start_s, 0), min(s.eeg_end_s, EEG_DURATION_S))

def ep_bp(df_roll, t0, t1):
    sub = df_roll[(df_roll['time_s']>=t0)&(df_roll['time_s']<=t1)]
    return {b: sub[b].mean() if len(sub) else np.nan for b in FREQ_BANDS}

ep_f = pd.DataFrame({n: ep_bp(df_roll_f, t0, t1) for n,(t0,t1) in epochs.items()}).T*100
ep_o = pd.DataFrame({n: ep_bp(df_roll_o, t0, t1) for n,(t0,t1) in epochs.items()}).T*100
print('Frontale Bandleistung pro Epoche (%):')
print(ep_f.round(1))

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.32,0.28,0.4],
    subplot_titles=['Frontale Bandleistung (rollend, 2s)',
                    'Engagement-Index β/[α+θ]',
                    'EEG Frontal – lokal normiert (1:10 downsampled)'])

for band, color in BAND_COLORS.items():
    if band=='Other' or band not in df_roll_f.columns: continue
    fig.add_trace(go.Scatter(x=df_roll_f['time_s'], y=df_roll_f[band],
        name=band, line=dict(color=color, width=1.5), legendgroup=band), row=1, col=1)

fig.add_trace(go.Scatter(x=df_roll_f['time_s'], y=df_roll_f['engagement'],
    name='Engagement', line=dict(color='#FFD600', width=1.5)), row=2, col=1)
fig.add_hline(y=1.0, line_dash='dot', line_color='#888', row=2, col=1)

ds = 10
fig.add_trace(go.Scatter(x=df_raw['time_s'][::ds], y=df_raw['frontal_norm'][::ds],
    name='Frontal (norm.)', line=dict(color='#E85D04', width=0.5), opacity=0.8), row=3, col=1)
fig.add_hline(y=0, line_dash='dot', line_color='#444', row=3, col=1)

vc_line = ['#2DCE89','#FFD600']
for i, s in enumerate(usable):
    for r in range(1,4):
        fig.add_vline(x=s.offset_s, line_dash='dash', line_color=vc_line[i%2],
            line_width=1.5, annotation_text=f'Video {i+1}' if r==1 else '', row=r, col=1)

fig.update_layout(height=700, hovermode='x unified',
    title=dict(text='EEG Zeitverlauf', font=dict(size=15)),
    legend=dict(orientation='h', y=-0.07))
fig.update_xaxes(title_text='Zeit ab EEG-Start (s)', row=3, col=1)
fig.write_html(f'{OUTPUT_DIR}/04_zeitverlauf.html')
fig.show()
print(f'Gespeichert: {OUTPUT_DIR}/04_zeitverlauf.html')

## 7. Spektralanalyse

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
for ax, (label, ch) in zip(axes, [('Frontal','frontal_f'),('Okzipital','okzipital_f')]):
    f_s, t_s, Sxx = spectrogram(df_raw[ch].values, fs=FS,
                                  nperseg=FS, noverlap=FS//2, nfft=FS*2)
    m   = (f_s>=0.5)&(f_s<=45)
    Sdb = 10*np.log10(Sxx[m]+1e-12)
    im  = ax.pcolormesh(t_s, f_s[m], Sdb, shading='gouraud', cmap='inferno',
                         vmin=np.percentile(Sdb,5), vmax=np.percentile(Sdb,98))
    for band,(lo,hi) in FREQ_BANDS.items():
        ax.axhline(lo, color=BAND_COLORS[band], lw=0.8, ls='--', alpha=0.7)
        ax.text(t_s[-1]*1.002, (lo+hi)/2, band, color=BAND_COLORS[band],
                fontsize=7.5, va='center', fontweight='bold')
    for i,s in enumerate(usable):
        ax.axvline(s.offset_s, color=vc_line[i%2], lw=1.5, ls='--')
    plt.colorbar(im, ax=ax, label='Leistung (dB)')
    ax.set_ylabel('Frequenz (Hz)')
    ax.set_title(f'{label} – Spektrogramm', fontweight='bold')
axes[1].set_xlabel('Zeit (s)')
plt.suptitle('EEG-Spektrogramm (0.5–45 Hz)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_spektrogramm.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Auffälligkeits-Erkennung: Lokaler Kontrast

### Methode: Local RMS Contrast

Statt absoluter z-Scores (die bei konstanter Gesamtamplitude schlecht funktionieren)  
wird der **lokale RMS-Kontrast** berechnet:

$$\text{contrast}(t) = \frac{\text{RMS}_{\text{kurz}}(t, 1\text{s})}{\text{RMS}_{\text{lang}}(t, 12\text{s})}$$

- **Kontrast ≫ 1**: Das Signal ist gerade deutlich stärker als sein eigener Umgebungspegel  
  → echter Ausschlag im Kontext
- **Kontrast ≈ 1**: Signal verhält sich wie die umliegenden Sekunden  
- **Gradient des Kontrastes**: Erkennt den *Onset* von Änderungen (plötzlicher Wechsel)

Zusätzlich werden Anomalien per **lokalem Band-Kontrast** erkannt:  
z-Score des rollenden Band-Power-Wertes innerhalb eines 30s-lokalen Fensters  
(statt globalem z-Score) → sensitiv für plötzliche Frequenzverschiebungen.

In [ ]:
def local_rms_contrast(signal, fs,
                        short_s=LOCAL_SHORT_S, long_s=LOCAL_LONG_S):
    """
    Verhältnis von kurzfristigem RMS zu langfristigem Umgebungs-RMS.
    Werte >> 1 = Signal gerade stärker als Umgebung (echter relativer Ausschlag).
    Der Gradient des Kontrastes erkennt Onsets von Änderungen.
    """
    sig_sq    = signal ** 2
    rms_short = np.sqrt(uniform_filter1d(sig_sq, size=int(short_s*fs)) + 1e-10)
    rms_long  = np.sqrt(uniform_filter1d(sig_sq, size=int(long_s*fs))  + 1e-10)
    contrast  = rms_short / rms_long
    return contrast


def local_band_zscore(df_roll, band, local_window_n=60):
    """
    Z-Score des Band-Power-Wertes relativ zu einem lokalen Fenster
    (statt globalem z-Score). Sensitiv für plötzliche Frequenzwechsel.
    local_window_n: Anzahl der Zeitpunkte im Referenzfenster (~30s bei step=0.5s)
    """
    x     = df_roll[band].values
    n     = local_window_n
    out   = np.zeros_like(x)
    for i in range(len(x)):
        lo = max(0, i - n)
        hi = min(len(x), i + n + 1)
        window = x[lo:hi]
        std    = window.std()
        if std > 1e-8:
            out[i] = (x[i] - window.mean()) / std
    return out


print('Berechne lokalen RMS-Kontrast...')
contrast = local_rms_contrast(df_raw['frontal_f'].values, FS,
                               short_s=LOCAL_SHORT_S, long_s=LOCAL_LONG_S)

# Gradient des Kontrastes (Onset-Detektion)
# Glätten mit 0.5s um hochfrequentes Rauschen zu entfernen
contrast_smooth = uniform_filter1d(contrast, size=int(0.5*FS))
contrast_grad   = np.abs(np.gradient(contrast_smooth, 1/FS))
# Nochmals glätten für stabilere Peaks
contrast_grad   = uniform_filter1d(contrast_grad, size=int(0.2*FS))

# Lokale Band-z-Scores
print('Berechne lokale Band-z-Scores...')
alpha_lz = local_band_zscore(df_roll_f, 'Alpha')
beta_lz  = local_band_zscore(df_roll_f, 'Beta')
theta_lz = local_band_zscore(df_roll_f, 'Theta')

print('Fertig.')
print(f'  Kontrast-Werte: min={contrast.min():.2f}, mean={contrast.mean():.2f}, max={contrast.max():.2f}')
print(f'  Gradient-Werte: 99. Pzt.={np.percentile(contrast_grad,99):.3f}')

## 9. Persistente Auffälligkeits-Perioden

Neben einzelnen Ereignissen sind **anhaltende Phasen erhöhter Aktivität** oft bedeutsamer.  
Hier werden Perioden gesucht, in denen der lokale RMS-Kontrast über mehrere Sekunden  
kontinuierlich erhöht ist (Threshold > 1.5, Mindestdauer 5s).  
Diese Methode ist ein einfaches Schwellenwert-Verfahren mit Mindestdauer, verwandt mit **CUSUM-artigen Kontrollkarten** (Cumulative Sum Control Chart) – im Unterschied zu echtem CUSUM wird hier aber keine kumulative Summe geführt, sondern direkt auf dem geglätteten Kontrastsignal geschwellt.

In [ ]:
def find_persistent_periods(contrast_signal, fs,
                              threshold=PERSIST_THRESHOLD,
                              min_duration_s=PERSIST_MIN_DURATION_S):
    """
    Findet Abschnitte, in denen der Kontrast dauerhaft über dem Threshold liegt.
    Gibt Liste von (t_start, t_end, mean_contrast) zurück.
    """
    above = (contrast_signal > threshold).astype(int)
    # Onset/Offset der Perioden
    diff       = np.diff(above, prepend=0, append=0)
    onsets     = np.where(diff ==  1)[0]
    offsets    = np.where(diff == -1)[0]
    min_samples = int(min_duration_s * fs)
    periods = []
    for on, off in zip(onsets, offsets):
        if off - on >= min_samples:
            t_start = on  / fs
            t_end   = off / fs
            mean_c  = contrast_signal[on:off].mean()
            periods.append({'t_start': t_start, 't_end': t_end,
                             'duration_s': t_end - t_start, 'mean_contrast': mean_c})
    if not periods:
        return pd.DataFrame(columns=['t_start', 't_end', 'duration_s', 'mean_contrast'])
    return pd.DataFrame(periods).sort_values('mean_contrast', ascending=False).reset_index(drop=True)


df_persist = find_persistent_periods(contrast_smooth, FS)
print(f'{len(df_persist)} persistente Phasen (Schwelle {PERSIST_THRESHOLD}, min. {PERSIST_MIN_DURATION_S}s):')
for _, row in df_persist.iterrows():
    vid_t = row['t_start'] - SYNC.offset_s
    vid_info = f'  [Video: {vid_t:.1f}s]' if 0 <= vid_t <= SYNC.video_duration_s else ''
    print(f'  t={row["t_start"]:.1f}s – {row["t_end"]:.1f}s  '
          f'({row["duration_s"]:.1f}s, ø Kontrast={row["mean_contrast"]:.2f}){vid_info}')

In [ ]:
# ── Top-Ereignisse ────────────────────────────────────────────────────
def detect_anomalies_local(contrast_grad, alpha_lz, beta_lz, theta_lz,
                             df_roll_f, fs,
                             n_top=N_TOP_ANOMALIES,
                             min_gap_s=MIN_ANOMALY_GAP_S):
    """
    Kombiniert mehrere lokal-kontrastbasierte Quellen zu einer
    einheitlichen Anomalie-Liste.
    Quellen:
      1. Gradient des RMS-Kontrastes (Onset plötzlicher Amplitudenänderung)
      2. Lokaler Alpha-z-Score (plötzlicher Alpha-Burst im Kontext)
      3. Lokaler Beta-z-Score (plötzlicher Beta-Spike im Kontext)
      4. Lokaler Theta-z-Score (plötzlicher Theta-Anstieg im Kontext)
      5. Engagement-Sprung (lokaler z-Score des Engagement-Index)
    """
    candidates = []

    # 1. Amplitude Onset
    thresh_grad = np.percentile(contrast_grad, 97)
    t_sig = np.arange(len(contrast_grad)) / fs
    above = contrast_grad > thresh_grad
    onsets = np.where(np.diff(above.astype(int)) == 1)[0]
    for idx in onsets:
        score = contrast_grad[idx] / thresh_grad
        candidates.append({'time_s': t_sig[idx], 'type': 'Amplituden-Onset',
                            'score': score, 'label': f'Plötzliche Amplitudenänderung (x{score:.1f})'})

    # 2–4. Lokale Band-Onsets
    t_roll = df_roll_f['time_s'].values
    for lz, atype, tlabel in [
        (alpha_lz, 'Alpha-Burst', 'Alpha-Burst (lokal z={:.1f})'),
        (beta_lz,  'Beta-Spike',  'Beta-Spike (lokal z={:.1f})'),
        (theta_lz, 'Theta-Burst', 'Theta-Burst (lokal z={:.1f})'),
    ]:
        above = lz > 2.0
        onsets = np.where(np.diff(above.astype(int)) == 1)[0]
        for idx in onsets:
            score = lz[idx]
            candidates.append({'time_s': float(t_roll[idx]), 'type': atype,
                                'score': score, 'label': tlabel.format(score)})

    # 5. Engagement-Sprung
    eng_lz = local_band_zscore(df_roll_f.assign(Eng=df_roll_f['engagement']), 'Eng')
    for sign, lbl in [(1, 'Engagement-Hoch'), (-1, 'Engagement-Tief')]:
        above = (eng_lz * sign) > 2.0
        onsets = np.where(np.diff(above.astype(int)) == 1)[0]
        for idx in onsets:
            score = abs(eng_lz[idx])
            candidates.append({'time_s': float(t_roll[idx]), 'type': lbl,
                                'score': score, 'label': f'{lbl} (z={score:.1f})'})

    if not candidates:
        return pd.DataFrame()

    df_c = pd.DataFrame(candidates).sort_values('score', ascending=False).reset_index(drop=True)
    # Non-maximum-Suppression
    selected, used = [], []
    for _, row in df_c.iterrows():
        t = row['time_s']
        if all(abs(t - u) >= min_gap_s for u in used):
            selected.append(row); used.append(t)
        if len(selected) >= n_top: break
    return pd.DataFrame(selected).sort_values('time_s').reset_index(drop=True)


df_anom = detect_anomalies_local(contrast_grad, alpha_lz, beta_lz, theta_lz, df_roll_f, FS)
print(f'{len(df_anom)} Top-Ereignisse (lokal-kontrastbasiert):')
for _, row in df_anom.iterrows():
    vid_t = row['time_s'] - SYNC.offset_s
    vi = f'  → Video: {vid_t:.1f}s' if 0 <= vid_t <= SYNC.video_duration_s else ''
    print(f'  t={row["time_s"]:.1f}s | {row["label"]}{vi}')

In [ ]:
# Übersichtsplot: Kontrast, Gradient, Anomalien
anom_colors = {
    'Amplituden-Onset': '#FF4B4B',
    'Alpha-Burst':      '#E85D04',
    'Beta-Spike':       '#1E88E5',
    'Theta-Burst':      '#45AAB8',
    'Engagement-Hoch':  '#2DCE89',
    'Engagement-Tief':  '#FFD600',
}

t_sig = df_raw['time_s'].values
fig, axes = plt.subplots(4, 1, figsize=(17, 11), sharex=True,
    gridspec_kw={'height_ratios': [2, 1.2, 1.2, 0.8]})

# EEG (normiert)
ds = 5
axes[0].plot(t_sig[::ds], df_raw['frontal_norm'].values[::ds],
             color='#666', lw=0.4, alpha=0.9)
axes[0].axhline(0, color='#333', lw=0.5)
axes[0].set_ylim(-5, 5)
axes[0].set_ylabel('Amp. (lokal norm.)')
axes[0].set_title('Frontales EEG (lokal normiert) mit Auffälligkeiten', fontweight='bold')

# Kontrast
axes[1].plot(t_sig, contrast_smooth, color='#888', lw=0.8)
axes[1].axhline(PERSIST_THRESHOLD, color='#FF4B4B', lw=1, ls='--',
                label=f'Persistenz-Schwelle ({PERSIST_THRESHOLD})')
axes[1].fill_between(t_sig, contrast_smooth, PERSIST_THRESHOLD,
    where=(contrast_smooth > PERSIST_THRESHOLD), alpha=0.3, color='#FF4B4B')
for _, p in df_persist.iterrows():
    axes[1].axvspan(p['t_start'], p['t_end'], alpha=0.15, color='#FF4B4B')
axes[1].set_ylabel('RMS-Kontrast')
axes[1].legend(fontsize=8)

# Gradient
axes[2].plot(t_sig, contrast_grad, color='#6C63FF', lw=0.6)
axes[2].set_ylabel('Kontrast-Gradient\n(Onset-Detektion)')

# Engagement
axes[3].plot(df_roll_f['time_s'], df_roll_f['engagement'],
             color='#FFD600', lw=0.9)
axes[3].axhline(1, color='#888', lw=0.6, ls=':')
axes[3].set_ylabel('Engagement')

# Marker für Auffälligkeiten und persistente Perioden
legend_patches = {}
for i, row in df_anom.iterrows():
    color = anom_colors.get(row['type'], '#fff')
    for ax in axes:
        ax.axvline(row['time_s'], color=color, lw=1.1, ls='--', alpha=0.8)
    axes[0].text(row['time_s'], 4.4, f'{i+1}', color=color,
                 fontsize=7, ha='center', fontweight='bold')
    if row['type'] not in legend_patches:
        legend_patches[row['type']] = mpatches.Patch(color=color, label=row['type'])

for ax in axes:
    for i, s in enumerate(usable):
        ax.axvline(s.offset_s, color=vc_line[i%2], lw=1.5, ls='-', alpha=0.4)

axes[3].set_xlabel('Zeit ab EEG-Start (s)')
axes[0].legend(handles=list(legend_patches.values()),
               loc='upper right', fontsize=7.5, ncol=3)
plt.suptitle('EEG-Übersicht: Lokaler Kontrast & Auffälligkeiten', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_anomalien.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Top-Auffälligkeiten mit Video-Screenshots

Für jedes Ereignis: EEG-Kontext (mit lokal normierter Skala + Amplitudenzonen) und Video-Frame.

In [ ]:
ANOM_INTERP = {
    'Amplituden-Onset': 'Plötzliche Amplitudenänderung relativ zur unmittelbaren Umgebung '
                        '– kein globaler Spike, sondern ein echter lokaler Wechsel. '
                        'Mögliche Ursache: Bewegungsartefakt, Erschrecken, oder kurze intense neuronale Reaktion.',
    'Alpha-Burst':      'Lokaler Anstieg der Alpha-Aktivität (8–13 Hz) im Vergleich zur '
                        'unmittelbaren Umgebung. Hinweis auf kurze Aufmerksamkeitsabnahme, '
                        'inneres Abschweifen oder einen Moment visueller Entspannung.',
    'Beta-Spike':       'Lokaler Anstieg der Beta-Aktivität (13–30 Hz) relativ zum Kontext. '
                        'Deutet auf erhöhte kognitive Anspannung oder einen überraschenden '
                        'Interface-Moment (z.B. unerwartete Darstellung, Entscheidungsdruck) hin.',
    'Theta-Burst':      'Lokaler Theta-Anstieg (4–8 Hz): mögliche kognitive Überlastung, '
                        'aktives mentales Nachdenken (z.B. Einordnung einer Information) '
                        'oder beginnende Ermüdung.',
    'Engagement-Hoch':  'Lokaler Sprung des Engagement-Index β/[α+θ] nach oben: '
                        'maximale kognitive Aktivierung im Vergleich zur Umgebung. '
                        'Interface-Moment, der volle Aufmerksamkeit beansprucht.',
    'Engagement-Tief':  'Lokaler Einbruch des Engagement-Index: kognitive Aktivierung '
                        'fällt unter den Umgebungspegel – passives Betrachten, '
                        'fehlende Interaktivität oder Ablenkung.',
}


def plot_anomaly_panel(row, idx, df_raw, df_roll_f, sync, context_s=12.0):
    t_ev  = row['time_s']
    t0    = max(0, t_ev - context_s)
    t1    = min(EEG_DURATION_S, t_ev + context_s)
    vid_t = t_ev - sync.offset_s
    in_video = 0 <= vid_t <= sync.video_duration_s
    color = anom_colors.get(row['type'], '#aaa')

    # Screenshot
    ss_path = f'{OUTPUT_DIR}/screenshots/anom_{idx+1:02d}.jpg'
    frame_idx = None
    if in_video:
        frame_idx, ok = sync.screenshot_at_eeg_time(t_ev, ss_path)
        if not ok: in_video = False

    n_cols = 2 if in_video else 1
    fig = plt.figure(figsize=(17, 5.5) if in_video else (12, 5.5))
    gs  = gridspec.GridSpec(2, n_cols, figure=fig,
                             width_ratios=[1.5, 1] if in_video else [1],
                             hspace=0.4, wspace=0.28)

    # ── EEG lokal normiert (oben links) ──────────────────────────
    ax_s = fig.add_subplot(gs[0, 0])
    m_sig = (df_raw['time_s']>=t0) & (df_raw['time_s']<=t1)
    t_ctx = df_raw.loc[m_sig, 'time_s'].values
    s_ctx = df_raw.loc[m_sig, 'frontal_norm'].values
    ax_s.axhspan(-1.5,  1.5, alpha=0.08, color='green')   # Normalbereich
    ax_s.axhspan( 1.5,  4.0, alpha=0.07, color='orange')
    ax_s.axhspan(-4.0, -1.5, alpha=0.07, color='orange')
    ax_s.plot(t_ctx, s_ctx, color='#666', lw=0.7)
    ax_s.axvline(t_ev, color=color, lw=2, ls='--')
    ax_s.axhline(0, color='#444', lw=0.5)
    ax_s.set_ylim(-4.5, 4.5)   # feste lokale σ-Skala
    ax_s.set_ylabel('Amp. (lokal norm. σ)')
    ax_s.set_xlabel('Zeit (s)')
    ax_s.set_title('EEG Frontal – lokal normiert (feste σ-Skala)', fontsize=9)
    ax_s.text(t_ev, 4.0, f'  {row["type"]}', color=color,
              fontsize=8.5, fontweight='bold')
    # Kontext-Kontrast einzeichnen
    t_sig_all = df_raw['time_s'].values
    m_ctr = (t_sig_all>=t0) & (t_sig_all<=t1)
    ax_c = ax_s.twinx()
    ax_c.plot(t_sig_all[m_ctr], contrast_smooth[m_ctr],
              color='#aaa', lw=0.8, alpha=0.6, ls=':')
    ax_c.axhline(PERSIST_THRESHOLD, color='#aaa', lw=0.5, ls=':')
    ax_c.set_ylabel('RMS-Kontrast', fontsize=7, color='#aaa')
    ax_c.tick_params(labelsize=6, colors='#aaa')

    # ── Alpha & Beta (unten links) ────────────────────────────────
    ax_b = fig.add_subplot(gs[1, 0])
    m_roll = (df_roll_f['time_s']>=t0) & (df_roll_f['time_s']<=t1)
    ax_b.plot(df_roll_f.loc[m_roll,'time_s'], df_roll_f.loc[m_roll,'Alpha']*100,
              color=BAND_COLORS['Alpha'], lw=1.2, label='Alpha')
    ax_b.plot(df_roll_f.loc[m_roll,'time_s'], df_roll_f.loc[m_roll,'Beta']*100,
              color=BAND_COLORS['Beta'],  lw=1.2, label='Beta')
    ax_b.plot(df_roll_f.loc[m_roll,'time_s'], df_roll_f.loc[m_roll,'Theta']*100,
              color=BAND_COLORS['Theta'], lw=1.0, ls='--', alpha=0.7, label='Theta')
    ax_b.axvline(t_ev, color=color, lw=2, ls='--')
    ax_b.set_ylabel('Rel. Leistung (%)')
    ax_b.set_xlabel('Zeit (s)')
    ax_b.legend(fontsize=8)
    ax_b.set_title('Frontale Bandleistung', fontsize=9)

    # ── Video-Screenshot (rechts) ─────────────────────────────────
    if in_video and os.path.exists(ss_path):
        ax_v = fig.add_subplot(gs[:, 1])
        img  = cv2.cvtColor(cv2.imread(ss_path), cv2.COLOR_BGR2RGB)
        ax_v.imshow(img)
        ax_v.axis('off')
        ax_v.set_title(f'Video-Frame @ {vid_t:.1f}s  (Frame {frame_idx})', fontsize=9)
        ax_v.text(0.5, -0.03, f'EEG-Zeit: {t_ev:.1f}s | Video-Zeit: {vid_t:.1f}s',
                  transform=ax_v.transAxes, ha='center', fontsize=7.5, color='#888')

    plt.suptitle(f'#{idx+1}: {row["label"]}  |  EEG-t = {t_ev:.1f}s',
                 fontsize=11, fontweight='bold', color=color)
    return fig


print(f'Erzeuge {len(df_anom)} Panels...')
for i, (_, row) in enumerate(df_anom.iterrows()):
    fig = plot_anomaly_panel(row, i, df_raw, df_roll_f, SYNC)
    fig.savefig(f'{OUTPUT_DIR}/screenshots/panel_{i+1:02d}.png',
                dpi=115, bbox_inches='tight')
    plt.show(); plt.close()

# Interpretationstabelle
print()
print('═'*68)
print('AUFFÄLLIGKEITEN – INTERPRETATION')
print('═'*68)
for i, (_, row) in enumerate(df_anom.iterrows()):
    vid_t = row['time_s'] - SYNC.offset_s
    in_v  = 0 <= vid_t <= SYNC.video_duration_s
    print(f'\n#{i+1}  {row["type"]} @ t={row["time_s"]:.1f}s  (Score: {row["score"]:.2f})')
    print(f'     {"Video-Zeit: "+str(round(vid_t,1))+"s" if in_v else "(Baseline-Phase, außerhalb Video)"}')
    txt = ANOM_INTERP.get(row['type'], '')
    words, line, lines_out = txt.split(), '', []
    for w in words:
        if len(line)+len(w)+1 > 64:
            lines_out.append(line); line = w
        else:
            line = (line+' '+w).strip()
    if line: lines_out.append(line)
    for l in lines_out: print(f'     {l}')
print(f'\n{"═"*68}')

## 11. Video-Overlay

Das EEG wird als Live-Panel über das Brauprozess-Video gelegt, synchron zum Zeitstempel.

- **Farbe**: Die EEG-Linie wechselt nur alle `COLOR_UPDATE_S` Sekunden zur Farbe des
  gerade dominanten Frequenzbandes → ruhige Darstellung statt ständigem Flimmern.
- **Skalierung**: Lokal normiertes Signal (lokale σ-Einheiten) statt globaler Perzentile.
  Grüner/orangefarbener/roter Hintergrund codiert die Amplitude relativ zur lokalen Umgebung.
- **Rendering**: `RENDER_SECONDS = None` → vollständiges Video, sonst nur die ersten N Sekunden
  (praktisch für schnelle Vorschau-Läufe während der Entwicklung).
- **Kompression**: OpenCV kann in dieser Umgebung nur den unkomprimierten `mp4v`-Codec
  schreiben (kein H.264-Encoder verfügbar). Deshalb wird das Rohvideo nach dem Rendern
  automatisch per `ffmpeg`/`libx264` nachkomprimiert (~5× kleiner, sichtbar kein
  Qualitätsverlust) – siehe `transcode_to_h264()` unten. Ist `ffmpeg` nicht installiert,
  bleibt die unkomprimierte `mp4v`-Datei erhalten.

In [ ]:
def hex_to_bgr(h):
    h = h.lstrip('#')
    r,g,b = int(h[0:2],16),int(h[2:4],16),int(h[4:6],16)
    return (b,g,r)

BAND_BGR = {k: hex_to_bgr(v) for k,v in BAND_COLORS.items()}

# Dominante Band-Farbe: pro Zeitfenster (COLOR_UPDATE_S), nicht per Segment
def build_band_color_lookup(df_roll_f, fps, total_frames, offset_s, update_s=COLOR_UPDATE_S):
    """
    Erstellt ein Array der Länge total_frames mit der Farbe (BGR) des dominanten Bandes.
    Wechselt nur alle update_s Sekunden → kein Flimmern.
    """
    frames_per_block = max(1, int(fps * update_s))
    roll_times = df_roll_f['time_s'].values
    colors = []
    for fi in range(0, total_frames, frames_per_block):
        eeg_t = fi / fps + offset_s
        ri    = np.searchsorted(roll_times, eeg_t)
        ri    = min(ri, len(df_roll_f)-1)
        row   = df_roll_f.iloc[ri]
        bp    = {b: row[b] for b in FREQ_BANDS if b in row.index}
        dom   = max(bp, key=bp.get) if bp else 'Other'
        color = BAND_BGR.get(dom, BAND_BGR['Other'])
        for _ in range(frames_per_block):
            colors.append(color)
    return colors[:total_frames]


def draw_overlay(frame, eeg_win, rolling_row, band_color,
                  video_t, eeg_t, oh=OVERLAY_HEIGHT, alpha=OVERLAY_ALPHA):
    H, W = frame.shape[:2]
    ov   = frame.copy()
    py   = H - oh
    cv2.rectangle(ov, (0, py), (W, H), (15, 15, 20), -1)
    frame = cv2.addWeighted(ov, alpha, frame, 1-alpha, 0)

    # ── 1. Signalverlauf – lokal normiert, STABILE Farbe ─────────
    sw  = int(W * 0.55)
    sh  = oh - 52
    sy0 = py + 14
    sy1 = sy0 + sh
    DISPLAY_SIGMA = 4.0    # feste Y-Achse: ±4σ
    yr  = 2 * DISPLAY_SIGMA

    # Hintergrund + Amplitudenzonen
    cv2.rectangle(frame, (10, sy0-5), (sw, sy1+5), (22, 22, 32), -1)
    zero_y = int(sy1 - DISPLAY_SIGMA/yr*sh)   # y-Pos für σ=0 (Mitte)
    # Grüner Normalbereich (±1.5σ)
    y_norm_top = max(sy0, int(sy1 - (DISPLAY_SIGMA+1.5)/yr*sh))
    y_norm_bot = min(sy1, int(sy1 - (DISPLAY_SIGMA-1.5)/yr*sh))
    cv2.rectangle(frame, (10, y_norm_top), (sw, y_norm_bot), (0, 35, 0), -1)
    # Nulllinie
    cv2.line(frame, (10, zero_y), (sw, zero_y), (60, 65, 75), 1)

    if 'frontal_norm' in eeg_win.columns and len(eeg_win) > 2:
        sig   = np.clip(eeg_win['frontal_norm'].values, -DISPLAY_SIGMA, DISPLAY_SIGMA)
        n_pts = len(sig)
        xsc   = (sw-20) / max(n_pts-1, 1)
        for ii in range(n_pts-1):
            x1 = int(10 + ii*xsc)
            x2 = int(10 + (ii+1)*xsc)
            y1 = max(sy0, min(sy1, int(sy1-(sig[ii]  +DISPLAY_SIGMA)/yr*sh)))
            y2 = max(sy0, min(sy1, int(sy1-(sig[ii+1]+DISPLAY_SIGMA)/yr*sh)))
            cv2.line(frame, (x1,y1), (x2,y2), band_color, 1)

        # σ-Level Linien
        for sv, lbl, clr in [(1.5,'1.5s',(50,120,50)), (2.5,'2.5s',(50,100,160))]:
            for sign in [1, -1]:
                ly = max(sy0, min(sy1, int(sy1-(sign*sv+DISPLAY_SIGMA)/yr*sh)))
                cv2.line(frame, (10,ly), (sw,ly), clr, 1)
                cv2.putText(frame, ('+' if sign>0 else '-')+lbl,
                            (sw-42, ly-2), cv2.FONT_HERSHEY_SIMPLEX, 0.26, clr, 1)

    dom_label = max({b: float(rolling_row[b]) for b in FREQ_BANDS if b in rolling_row.index},
                     key=lambda b: float(rolling_row[b])) if rolling_row is not None else '?'
    cv2.putText(frame, f'EEG Frontal (lokal norm. +/-{DISPLAY_SIGMA:.0f} sigma) | Dom.: {dom_label}',
                (14, sy0-7), cv2.FONT_HERSHEY_SIMPLEX, 0.31, (140,140,140), 1)

    # ── 2. Frequenzband-Balken ────────────────────────────────────
    bx0  = sw + 18
    bw   = int(W * 0.22)
    bslt = (oh-52) // len(FREQ_BANDS)
    cv2.putText(frame, 'FREQUENZBAENDER', (bx0, py+15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.35, (175,175,175), 1)
    for bi, band in enumerate(FREQ_BANDS):
        pwr   = float(rolling_row[band]) if rolling_row is not None and band in rolling_row.index else 0.0
        by0   = py+23+bi*bslt; by1 = by0+bslt-4
        fx    = min(bx0+60+int(pwr*(bw-66)), bx0+bw-4)
        color = BAND_BGR.get(band, BAND_BGR['Other'])
        cv2.rectangle(frame, (bx0+60, by0+4), (bx0+bw-4, by1-4), (40,40,52), -1)
        if fx > bx0+60:
            cv2.rectangle(frame, (bx0+60, by0+4), (fx, by1-4), color, -1)
        cv2.putText(frame, band[:5], (bx0, (by0+by1)//2+4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)
        cv2.putText(frame, f'{pwr*100:.0f}%', (bx0+bw+2, (by0+by1)//2+4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.31, (135,135,135), 1)

    # ── 3. Engagement-Meter ───────────────────────────────────────
    ex0 = bx0+bw+16; ew = W-ex0-10
    cv2.putText(frame, 'ENGAGEMENT', (ex0, py+15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.33, (175,175,175), 1)
    eng = 0.5
    if rolling_row is not None:
        a  = float(rolling_row.get('Alpha', 0.01))
        b  = float(rolling_row.get('Beta',  0.01))
        th = float(rolling_row.get('Theta', 0.01))
        eng = min(1.0, b/max(a+th,0.001))
    my0, my1 = py+23, H-34
    cx  = ex0+ew//2; bh2 = max(ew//4, 8)
    cv2.rectangle(frame, (cx-bh2,my0), (cx+bh2,my1), (32,32,44), -1)
    fh = int(eng*(my1-my0))
    if fh > 0:
        er = int(min(255,eng*2*255)); eg = int(min(255,(1-eng)*2*255))
        cv2.rectangle(frame, (cx-bh2, my1-fh), (cx+bh2,my1), (0,eg,er), -1)
        cv2.putText(frame, f'{eng*100:.0f}%',
                    (cx-16, my1-fh-5 if fh<(my1-my0)-20 else my0+16),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.41, (205,205,205), 1)

    # ── 4. Zeitstempel ────────────────────────────────────────────
    cv2.putText(frame, f'EEG: {eeg_t:.1f}s | Video: {video_t:.1f}s',
                (10, H-12), cv2.FONT_HERSHEY_SIMPLEX, 0.34, (95,95,95), 1)
    return frame


print('Overlay-Funktion bereit.')

In [ ]:
import shutil
import subprocess


def transcode_to_h264(raw_path, final_path, crf=23, preset='medium'):
    """
    Komprimiert ein mit OpenCV (mp4v) geschriebenes Rohvideo per ffmpeg/libx264
    nach, ohne die Overlay-Pixel neu zeichnen zu müssen. mp4v ist in dieser
    OpenCV/ffmpeg-Umgebung der einzig verfügbare Schreib-Codec und praktisch
    unkomprimiert (Faktor ~5 größer als nötig). Ist ffmpeg nicht verfügbar,
    bleibt schlicht die Rohdatei als Ergebnis stehen.
    """
    if not shutil.which('ffmpeg'):
        print('  ⚠ ffmpeg nicht gefunden – belasse unkomprimiertes mp4v-Video.')
        return raw_path
    try:
        subprocess.run(
            ['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
             '-i', raw_path, '-c:v', 'libx264', '-crf', str(crf),
             '-preset', preset, '-pix_fmt', 'yuv420p', final_path],
            check=True)
        os.remove(raw_path)
        return final_path
    except subprocess.CalledProcessError as e:
        print(f'  ⚠ ffmpeg-Transcode fehlgeschlagen ({e}) – belasse mp4v-Video.')
        return raw_path


def render_overlay_video(sync, output_path, df_eeg, df_roll,
                          render_seconds=RENDER_SECONDS, codec='mp4v'):
    """
    Rendert das EEG-Overlay-Video und komprimiert es anschließend per ffmpeg.
    render_seconds: None = vollständig, Zahl = nur die ersten N Sekunden.
    """
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    raw_path = output_path.replace('.mp4', '_raw.mp4')
    cap = cv2.VideoCapture(sync.video_path)
    nf  = (sync.total_frames if render_seconds is None
            else min(int(sync.fps * render_seconds), sync.total_frames))
    out = cv2.VideoWriter(raw_path, cv2.VideoWriter_fourcc(*codec),
                           sync.fps, (sync.width, sync.height))

    dur = nf / sync.fps
    mode_str = 'vollständig' if render_seconds is None else f'erste {render_seconds}s'
    print(f'Render ({mode_str}): {sync.name}')
    print(f'  → {output_path}')
    print(f'  {nf:,} Frames | {sync.fps:.0f} fps | {dur:.0f}s | {sync.width}×{sync.height}')

    # Band-Farb-Lookup (stabile Farbe pro 2s-Block)
    band_colors_lut = build_band_color_lookup(df_roll, sync.fps, nf,
                                               sync.offset_s, COLOR_UPDATE_S)

    roll_times = df_roll['time_s'].values

    for fi in range(nf):
        ret, frame = cap.read()
        if not ret:
            print(f'  Frame {fi} nicht lesbar – Abbruch.'); break

        vid_t = fi / sync.fps
        eeg_t = vid_t + sync.offset_s
        win   = sync.eeg_window_at_frame(fi, df_eeg)

        ri   = np.searchsorted(roll_times, eeg_t)
        rrow = df_roll.iloc[min(ri, len(df_roll)-1)] if len(df_roll) else None

        frame = draw_overlay(frame, win, rrow, band_colors_lut[fi], vid_t, eeg_t)
        out.write(frame)

        if fi % 1000 == 0:
            print(f'  [{fi/nf*100:.0f}%] Frame {fi:,}/{nf:,}  (Video: {vid_t:.0f}s)')

    cap.release(); out.release()

    print('  Komprimiere per ffmpeg (libx264)...')
    final_path = transcode_to_h264(raw_path, output_path)
    size_mb = os.path.getsize(final_path)/1024/1024
    print(f'  ✓ {final_path}  ({size_mb:.0f} MB)')


print(f'Render-Konfiguration: RENDER_SECONDS = {RENDER_SECONDS}')
print(f'  → {"vollständiges Rendering" if RENDER_SECONDS is None else str(RENDER_SECONDS)+"s Vorschau"}')
print()

for s in usable:
    out_name = f'{OUTPUT_DIR}/eeg_overlay_{Path(s.video_path).stem}.mp4'
    render_overlay_video(s, out_name, df_raw, df_roll_f)
    print()

## 12. Dashboard & Erkenntnisse

In [ ]:
fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.48, wspace=0.35)

# PSD
for ci, (label, fq, ps, bp) in enumerate([
    ('Frontal', freqs_f, psd_f, bp_f), ('Okzipital', freqs_o, psd_o, bp_o)]):
    ax = fig.add_subplot(gs[0, ci*2:ci*2+2])
    ax.semilogy(fq[fq<=45], ps[fq<=45], color='#333', lw=1)
    for b,(lo,hi) in FREQ_BANDS.items():
        m = (fq>=lo)&(fq<=hi)
        ax.fill_between(fq, ps, where=m, alpha=0.4, color=BAND_COLORS[b],
                         label=f'{b} {bp[b]*100:.1f}%')
    ax.set_xlim(0.5,45); ax.set_xlabel('Hz'); ax.set_ylabel('PSD (log)')
    ax.set_title(f'{label} – Spektrum', fontweight='bold')
    ax.legend(fontsize=7, ncol=5)

# Epochenvergleich
ax3 = fig.add_subplot(gs[1, :2])
bands = list(FREQ_BANDS.keys()); x = np.arange(len(bands))
n_ep = len(ep_f); w = 0.75/max(n_ep,1)
for i,(ename,erow) in enumerate(ep_f.iterrows()):
    ax3.bar(x+i*w-(n_ep-1)*w/2, [erow.get(b,0) for b in bands],
            width=w*0.9, color=['#888','#E85D04','#1E88E5'][i%3], label=ename, alpha=0.85)
ax3.set_xticks(x); ax3.set_xticklabels(bands)
ax3.set_ylabel('Rel. Leistung (%)'); ax3.legend(fontsize=7)
ax3.set_title('Epochenvergleich Frontal', fontweight='bold')

# Persistente Perioden
ax4 = fig.add_subplot(gs[1, 2:])
ax4.plot(df_raw['time_s'][::5], contrast_smooth[::5], color='#888', lw=0.7)
ax4.axhline(PERSIST_THRESHOLD, color='#FF4B4B', lw=1, ls='--')
ax4.fill_between(df_raw['time_s'][::5], contrast_smooth[::5], PERSIST_THRESHOLD,
    where=(contrast_smooth[::5]>PERSIST_THRESHOLD), alpha=0.3, color='#FF4B4B')
for _, p in df_persist.head(3).iterrows():
    ax4.axvspan(p['t_start'], p['t_end'], alpha=0.25, color='#FF4B4B')
    ax4.text((p['t_start']+p['t_end'])/2, PERSIST_THRESHOLD*1.05,
             f'{p["duration_s"]:.0f}s', ha='center', fontsize=7, color='#FF4B4B')
for i,s in enumerate(usable):
    ax4.axvline(s.offset_s, color=vc_line[i%2], lw=1.5, ls='--')
ax4.set_xlabel('Zeit (s)'); ax4.set_ylabel('RMS-Kontrast')
ax4.set_title('Lokaler RMS-Kontrast & persistente Phasen', fontweight='bold')

# Pie Charts
for ci,(label,bp) in enumerate([('Frontal',bp_f),('Okzipital',bp_o)]):
    ax = fig.add_subplot(gs[2, ci*2:ci*2+2])
    ax.pie([bp[b] for b in FREQ_BANDS],
           labels=[f'{b}\n{bp[b]*100:.1f}%' for b in FREQ_BANDS],
           colors=[BAND_COLORS[b] for b in FREQ_BANDS],
           wedgeprops=dict(width=0.55, edgecolor='#0a0a0a'),
           startangle=90, textprops=dict(fontsize=8))
    ax.set_title(f'{label} – Bandverteilung', fontweight='bold')

plt.suptitle('EEG-Dashboard – BrewVis InfoVis', fontsize=14, fontweight='bold', y=1.01)
plt.savefig(f'{OUTPUT_DIR}/07_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Erkenntnisse ─────────────────────────────────────────────────────
ba_ratio  = bp_f['Beta'] / max(bp_f['Alpha'], 0.001)
mean_eng  = df_roll_f['engagement'].mean()
ep_names  = list(ep_f.index)
base_a    = ep_f.loc[ep_names[0],'Alpha'] if len(ep_f)>0 else 0
vid_a     = ep_f.loc[ep_names[1],'Alpha'] if len(ep_f)>1 else base_a
base_b    = ep_f.loc[ep_names[0],'Beta']  if len(ep_f)>0 else 0
vid_b     = ep_f.loc[ep_names[1],'Beta']  if len(ep_f)>1 else base_b

print('════════════════════════════════════════════════════════════')
print('ERKENNTNISSE – BrewVis EEG-Session vom 10.06.2026')
print('════════════════════════════════════════════════════════════')
print(f'\nAufnahme: {EEG_T0} → {EEG_END}  ({EEG_DURATION_S:.0f}s)')
print(f'\nGlobale Bandleistung (Frontal):')
for b in FREQ_BANDS:
    bar = '█'*int(bp_f[b]*40)
    print(f'  {b:<6} {bp_f[b]*100:>5.1f}%  {bar}')
print(f'\nEngagement β/α: {ba_ratio:.2f}  |  Mittl. Engagement-Index: {mean_eng:.2f}')
level = 'HOCH' if ba_ratio>1.5 else ('MITTEL' if ba_ratio>0.8 else 'NIEDRIG')
print(f'  → {level}: ', end='')
if ba_ratio>1.5: print('starke kognitive Aktivierung während der Session.')
elif ba_ratio>0.8: print('ausgewogene Aktivierung, Balance Fokus/Entspannung.')
else: print('eher passiver/entspannter Zustand.')

ac = (vid_a/base_a-1)*100 if base_a>0 else 0
bc = (vid_b/base_b-1)*100 if base_b>0 else 0
print(f'\nEpochenvergleich Baseline → Video:')
print(f'  Alpha: {base_a:.1f}% → {vid_a:.1f}%  ({"▲" if ac>0 else "▼"}{abs(ac):.0f}%)')
print(f'  Beta:  {base_b:.1f}% → {vid_b:.1f}%  ({"▲" if bc>0 else "▼"}{abs(bc):.0f}%)')
print(f'\nPersistente Aktivitätsphasen: {len(df_persist)}')
for _, p in df_persist.head(5).iterrows():
    vt = p["t_start"]-SYNC.offset_s
    vi = f'  [Video ~{vt:.0f}s]' if 0<=vt<=SYNC.video_duration_s else ''
    print(f'  {p["t_start"]:.1f}–{p["t_end"]:.1f}s  ({p["duration_s"]:.0f}s, ø Kontrast={p["mean_contrast"]:.2f}){vi}')
print(f'\nTop-Ereignisse: {len(df_anom)}')
for _, r in df_anom.iterrows():
    print(f'  {r["type"]:<20} t={r["time_s"]:.1f}s')
print(f'\nDesign-Empfehlungen (aus EEG-Daten abgeleitet):')
print(f'  1. β/α = {ba_ratio:.2f} – Interface erzeugt aktive kognitive Einbindung (positiv).')
if ac > 15:
    print(f'  2. Alpha +{ac:.0f}% während Video → mehr Interaktivität einbauen.')
elif ac < -10:
    print(f'  2. Alpha -{abs(ac):.0f}% → erhöhte visuelle Aufmerksamkeit, gute Visualisierung.')
if len(df_persist) > 0:
    top_per = df_persist.iloc[0]
    print(f'  3. Persistente Phase {top_per["t_start"]:.0f}–{top_per["t_end"]:.0f}s prüfen')
    print(f'     → anhaltend erhöhte Aktivität (Stress? Überraschung? starkes Engagement?)')
print(f'  4. Okzipitale Delta-Dominanz ({bp_o["Delta"]*100:.0f}%): visuell komplexe Darstellung –')
print(f'     Kontrast und Lesbarkeit der Labels prüfen.')
anom_types = df_anom['type'].value_counts()
if 'Theta-Burst' in anom_types:
    print(f'  5. {anom_types["Theta-Burst"]}× Theta-Burst → kognitive Überlastungsmomente –')
    print(f'     Informationsdichte der entsprechenden Ansichten reduzieren.')
print('════════════════════════════════════════════════════════════')

## 13. Erweiterte Analysen

Jetzt gehen wir tiefer: Wir untersuchen, **ob und wie sich die Aufmerksamkeit über die Zeit verändert**  
und ob bestimmte Phasen der Datendarstellung besser oder schlechter funktionieren.

Diese Analysen zielen direkt auf die Studienarbeit-Frage ab:  
*"Ist die Visualisierung der Brauprozess-Daten gut gestaltet?"*

In [ ]:
# ── 14.1 Aufmerksamkeits-Timeline ────────────────────────────────────
# Wir glätten den Engagement-Index stark, um den groben Verlauf der
# Aufmerksamkeit über die gesamte Session zu sehen (Trend statt Zappeln).

from scipy.ndimage import uniform_filter1d as _uf

# Engagement glätten (30s gleitender Durchschnitt)
eng_raw    = df_roll_f['engagement'].values
window_pts = int(30 / 0.5)   # 30s bei 0.5s Schrittweite
eng_smooth = _uf(eng_raw, size=window_pts)
t_roll     = df_roll_f['time_s'].values

# Aufmerksamkeit in 3 Kategorien einteilen (relativ zum eigenen Mittel)
eng_median = np.median(eng_smooth)
eng_std    = np.std(eng_smooth)
high_thresh = eng_median + 0.5 * eng_std
low_thresh  = eng_median - 0.5 * eng_std

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(t_roll, eng_raw, color='#ddd', lw=0.5, alpha=0.5, label='Engagement (roh)')
ax.plot(t_roll, eng_smooth, color='#1E88E5', lw=2.5, label='Engagement (30s geglättet)')

# Farbige Bereiche
ax.fill_between(t_roll, eng_smooth, high_thresh, where=(eng_smooth >= high_thresh),
                alpha=0.3, color='#2DCE89', label='Hohe Aufmerksamkeit')
ax.fill_between(t_roll, eng_smooth, low_thresh, where=(eng_smooth <= low_thresh),
                alpha=0.3, color='#FF4B4B', label='Niedrige Aufmerksamkeit')
ax.axhline(eng_median, color='#888', ls=':', lw=1, label='Median')

# Video-Marker
for i, s in enumerate(usable):
    ax.axvline(s.offset_s, color=vc_line[i%2], lw=2, ls='--')
    ax.text(s.offset_s, ax.get_ylim()[1]*0.95, f' Video {i+1} Start',
            fontsize=9, color=vc_line[i%2], fontweight='bold')

ax.set_xlabel('Zeit ab EEG-Start (s)')
ax.set_ylabel('Engagement-Index (β/[α+θ])')
ax.set_title('Aufmerksamkeits-Verlauf über die Session', fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/08_aufmerksamkeit_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# Textauswertung
high_pct = (eng_smooth >= high_thresh).mean() * 100
low_pct  = (eng_smooth <= low_thresh).mean() * 100
print(f'Anteil hoher Aufmerksamkeit:    {high_pct:.0f}% der Zeit')
print(f'Anteil niedriger Aufmerksamkeit: {low_pct:.0f}% der Zeit')
print(f'Anteil mittlerer Aufmerksamkeit: {100-high_pct-low_pct:.0f}% der Zeit')

In [ ]:
# ── 14.2 Verändert sich die Aufmerksamkeit im Laufe des Videos? ───────
# Wichtige Design-Frage: Lässt die Aufmerksamkeit nach (Ermüdung/Langeweile)
# oder bleibt sie stabil? Wir teilen die Video-Phase in Viertel.

video_start = SYNC.offset_s
video_end   = min(SYNC.eeg_end_s, EEG_DURATION_S)
video_dur   = video_end - video_start

quarters = []
for q in range(4):
    q0 = video_start + q * video_dur / 4
    q1 = video_start + (q+1) * video_dur / 4
    mask = (t_roll >= q0) & (t_roll < q1)
    quarters.append({
        'Viertel': f'Q{q+1}',
        'Zeit': f'{q0:.0f}-{q1:.0f}s',
        'Engagement': eng_raw[mask].mean(),
        'Alpha_%': df_roll_f.loc[mask, 'Alpha'].mean() * 100,
        'Beta_%':  df_roll_f.loc[mask, 'Beta'].mean() * 100,
        'Theta_%': df_roll_f.loc[mask, 'Theta'].mean() * 100,
    })

df_quarters = pd.DataFrame(quarters)
print('Aufmerksamkeit über die Video-Viertel:')
print(df_quarters.round(2).to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Engagement-Trend
x_q = range(len(df_quarters))
ax1.plot(x_q, df_quarters['Engagement'], 'o-', color='#1E88E5', lw=2, markersize=10)
ax1.set_xticks(x_q)
ax1.set_xticklabels([f'{r["Viertel"]}\n{r["Zeit"]}' for _, r in df_quarters.iterrows()])
ax1.set_ylabel('Mittlerer Engagement-Index')
ax1.set_title('Aufmerksamkeits-Trend (Video-Viertel)', fontweight='bold')
ax1.grid(alpha=0.3)
# Trendlinie
z = np.polyfit(x_q, df_quarters['Engagement'], 1)
ax1.plot(x_q, np.poly1d(z)(x_q), '--', color='#FF4B4B', lw=1.5,
         label=f'Trend: {"fallend" if z[0]<0 else "steigend"}')
ax1.legend(fontsize=9)

# Band-Entwicklung
for band in ['Alpha', 'Beta', 'Theta']:
    ax2.plot(x_q, df_quarters[f'{band}_%'], 'o-', color=BAND_COLORS[band],
             lw=2, markersize=8, label=band)
ax2.set_xticks(x_q)
ax2.set_xticklabels([r['Viertel'] for _, r in df_quarters.iterrows()])
ax2.set_ylabel('Rel. Leistung (%)')
ax2.set_title('Frequenzbänder über die Video-Viertel', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/09_viertel_trend.png', dpi=150, bbox_inches='tight')
plt.show()

# Interpretation
eng_trend = z[0]
print()
if eng_trend < -0.02:
    print('📉 Aufmerksamkeit NIMMT AB über das Video.')
    print('   → Mögliche Ermüdung oder nachlassendes Interesse.')
    print('   → Design: Video evtl. kürzen oder Abwechslung/Höhepunkte einbauen.')
elif eng_trend > 0.02:
    print('📈 Aufmerksamkeit NIMMT ZU über das Video.')
    print('   → Inhalt wird interessanter / Person "taut auf".')
    print('   → Design: guter Spannungsaufbau.')
else:
    print('➡️  Aufmerksamkeit bleibt STABIL über das Video.')
    print('   → Gleichmäßige Einbindung, keine Ermüdung erkennbar (positiv).')

In [ ]:
# ── 14.3 Wie hängen Denken (frontal) und Sehen (okzipital) zusammen? ──
# Wenn die visuelle Verarbeitung (okzipital) und die Aufmerksamkeit (frontal)
# zusammen ansteigen, deutet das auf gute Aufmerksamkeitslenkung durch das Bild hin.

# Beide auf gleiche Zeitachse bringen
min_len = min(len(df_roll_f), len(df_roll_o))
frontal_beta  = df_roll_f['Beta'].values[:min_len]
okzipital_beta = df_roll_o['Beta'].values[:min_len]
frontal_alpha  = df_roll_f['Alpha'].values[:min_len]
okzipital_alpha = df_roll_o['Alpha'].values[:min_len]

corr_beta  = np.corrcoef(frontal_beta, okzipital_beta)[0, 1]
corr_alpha = np.corrcoef(frontal_alpha, okzipital_alpha)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].scatter(frontal_beta*100, okzipital_beta*100, s=8, alpha=0.4, color='#1E88E5')
axes[0].set_xlabel('Frontal Beta (%)')
axes[0].set_ylabel('Okzipital Beta (%)')
axes[0].set_title(f'Beta-Kopplung (r={corr_beta:.2f})', fontweight='bold')
z1 = np.polyfit(frontal_beta*100, okzipital_beta*100, 1)
xr = np.array([frontal_beta.min()*100, frontal_beta.max()*100])
axes[0].plot(xr, np.poly1d(z1)(xr), '--', color='#FF4B4B', lw=1.5)

axes[1].scatter(frontal_alpha*100, okzipital_alpha*100, s=8, alpha=0.4, color='#E85D04')
axes[1].set_xlabel('Frontal Alpha (%)')
axes[1].set_ylabel('Okzipital Alpha (%)')
axes[1].set_title(f'Alpha-Kopplung (r={corr_alpha:.2f})', fontweight='bold')
z2 = np.polyfit(frontal_alpha*100, okzipital_alpha*100, 1)
xr2 = np.array([frontal_alpha.min()*100, frontal_alpha.max()*100])
axes[1].plot(xr2, np.poly1d(z2)(xr2), '--', color='#FF4B4B', lw=1.5)
plt.suptitle('Zusammenhang Frontal (Denken) ↔ Okzipital (Sehen)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/10_korrelation.png', dpi=150, bbox_inches='tight')
plt.show()

def interpret_corr(r):
    a = abs(r)
    if a > 0.6: return 'stark'
    if a > 0.3: return 'mittel'
    if a > 0.1: return 'schwach'
    return 'kaum'

print(f'Beta-Korrelation (Denken ↔ Sehen):  r = {corr_beta:.2f} ({interpret_corr(corr_beta)})')
print(f'Alpha-Korrelation:                  r = {corr_alpha:.2f} ({interpret_corr(corr_alpha)})')
print()
print('Was bedeutet das?')
if corr_beta > 0.3:
    print('  Frontale und okzipitale Aktivität steigen gemeinsam → was die Person')
    print('  SIEHT, lenkt auch ihre AUFMERKSAMKEIT. Das spricht für eine gut')
    print('  gestaltete Visualisierung, die den Blick führt.')
else:
    print('  Schwacher Zusammenhang → visuelle Reize und Aufmerksamkeit laufen')
    print('  eher unabhängig. Die Darstellung lenkt die Aufmerksamkeit weniger gezielt.')

In [ ]:
# ── 14.4 Ist der Unterschied Baseline ↔ Video echt? (Statistik) ──────
# Wir prüfen, ob sich die Aufmerksamkeit wirklich messbar ändert, wenn das
# Video startet – oder ob es nur Zufallsschwankung ist.

from scipy.stats import mannwhitneyu

# Engagement-Werte: Baseline vs Video
base_mask  = t_roll < SYNC.offset_s
video_mask = (t_roll >= SYNC.offset_s) & (t_roll <= min(SYNC.eeg_end_s, EEG_DURATION_S))

eng_baseline = eng_raw[base_mask]
eng_video    = eng_raw[video_mask]

# Mann-Whitney-U-Test (kein Normalverteilungs-Zwang, robust)
if len(eng_baseline) > 5 and len(eng_video) > 5:
    stat, pval = mannwhitneyu(eng_baseline, eng_video, alternative='two-sided')
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bp = ax.boxplot([eng_baseline, eng_video], labels=['Baseline\n(vor Video)', 'Während Video'],
                     patch_artist=True, widths=0.6)
    for patch, color in zip(bp['boxes'], ['#888', '#1E88E5']):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_ylabel('Engagement-Index')
    ax.set_title('Aufmerksamkeit: Baseline vs. Video', fontweight='bold')
    
    # Signifikanz anzeigen
    sig_text = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'n.s.'))
    ymax = max(eng_baseline.max(), eng_video.max())
    ax.plot([1, 2], [ymax*1.05, ymax*1.05], color='black', lw=1)
    ax.text(1.5, ymax*1.08, sig_text, ha='center', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/11_baseline_vs_video.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'Mittlerer Engagement Baseline: {eng_baseline.mean():.2f}')
    print(f'Mittlerer Engagement Video:    {eng_video.mean():.2f}')
    print(f'Statistischer Test (Mann-Whitney-U): p = {pval:.4f}')
    print()
    if pval < 0.05:
        direction = 'höher' if eng_video.mean() > eng_baseline.mean() else 'niedriger'
        print(f'✓ Der Unterschied ist statistisch SIGNIFIKANT (p < 0.05).')
        print(f'  Die Aufmerksamkeit ist während des Videos messbar {direction} als davor.')
        if direction == 'höher':
            print(f'  → Das Video/die Visualisierung aktiviert die Person stärker. Gut!')
        else:
            print(f'  → Die Person war vor dem Video aktiver. Evtl. Visualisierung zu passiv.')
    else:
        print(f'✗ Der Unterschied ist NICHT signifikant (p ≥ 0.05).')
        print(f'  Baseline und Video unterscheiden sich nicht eindeutig –')
        print(f'  könnte Zufall sein. Bei nur 1 Person ist das aber normal.')
    print()
    print('Hinweis: "Signifikant" heißt: der Unterschied ist wahrscheinlich echt')
    print('und nicht nur Zufall. *** = sehr sicher, * = einigermaßen sicher,')
    print('n.s. = nicht sicher unterscheidbar.')
else:
    print('Zu wenig Daten für statistischen Vergleich.')
    pval = None

## 14. Ergebnis-Tabelle (alle Erkenntnisse)

Alle wichtigen Kennzahlen und Erkenntnisse in einer übersichtlichen Tabelle gesammelt –  
diese wird auch in den PDF-Report übernommen.

In [ ]:
# ── Alle Erkenntnisse in einem DataFrame sammeln ─────────────────────

results = []

def add(kategorie, kennzahl, wert, einheit, interpretation):
    results.append({
        'Kategorie': kategorie, 'Kennzahl': kennzahl,
        'Wert': wert, 'Einheit': einheit, 'Interpretation': interpretation
    })

# Aufnahme
add('Aufnahme', 'Dauer', f'{EEG_DURATION_S:.0f}', 's', f'{EEG_DURATION_S/60:.1f} Minuten')
add('Aufnahme', 'Sampling-Rate', f'{FS}', 'Hz', 'Messpunkte pro Sekunde')
add('Aufnahme', 'Nutzbare Videos', f'{len(usable)}/{len(all_syncs)}', '', 'durch EEG abgedeckt')

# Frequenzbänder Frontal
for b in FREQ_BANDS:
    interp = {
        'Delta': 'langsame Wellen (Schlaf/Artefakte)',
        'Theta': 'Konzentration oder Müdigkeit',
        'Alpha': 'Entspannung/Abschweifen',
        'Beta':  'aktives Denken/Fokus',
        'Gamma': 'intensive Verarbeitung',
    }[b]
    add('Frequenzbänder (Frontal)', b, f'{bp_f[b]*100:.1f}', '%', interp)

# Kennzahlen
add('Kennzahlen', 'Beta/Alpha-Verhältnis', f'{ba_ratio:.2f}', '',
    'HOCH (>1.5): aktiviert' if ba_ratio>1.5 else ('MITTEL' if ba_ratio>0.8 else 'NIEDRIG: passiv'))
add('Kennzahlen', 'Engagement-Index (Mittel)', f'{mean_eng:.2f}', '',
    'kognitive Einbindung über die Session')
add('Kennzahlen', 'Dominantes Band Frontal', max(bp_f, key=bp_f.get), '',
    'stärkste Gehirnwelle im Stirnbereich')
add('Kennzahlen', 'Dominantes Band Okzipital', max(bp_o, key=bp_o.get), '',
    'stärkste Gehirnwelle im Sehzentrum')

# Aufmerksamkeitsverlauf
trend_txt = 'fallend (Ermüdung?)' if eng_trend < -0.02 else ('steigend (gut)' if eng_trend > 0.02 else 'stabil (gut)')
add('Aufmerksamkeit', 'Trend über Video', trend_txt, '', f'Steigung: {eng_trend:+.3f}')
add('Aufmerksamkeit', 'Hohe Aufmerksamkeit', f'{high_pct:.0f}', '%', 'Anteil der Zeit')
add('Aufmerksamkeit', 'Niedrige Aufmerksamkeit', f'{low_pct:.0f}', '%', 'Anteil der Zeit')

# Epochenvergleich
add('Baseline→Video', 'Alpha-Änderung', f'{ac:+.0f}', '%',
    'sinkt=mehr Fokus' if ac<0 else 'steigt=weniger Fokus')
add('Baseline→Video', 'Beta-Änderung', f'{bc:+.0f}', '%',
    'steigt=mehr Aktivität' if bc>0 else 'sinkt=weniger Aktivität')
if pval is not None:
    add('Baseline→Video', 'Statistik (p-Wert)', f'{pval:.3f}', '',
        'signifikant' if pval<0.05 else 'nicht signifikant (bei N=1 normal)')

# Kopplung
add('Elektroden-Kopplung', 'Beta-Korrelation', f'{corr_beta:.2f}', 'r',
    f'{interpret_corr(corr_beta)}er Zusammenhang Sehen↔Denken')

# Auffälligkeiten
add('Auffälligkeiten', 'Einzelereignisse', f'{len(df_anom)}', '', 'markante EEG-Momente')
add('Auffälligkeiten', 'Persistente Phasen', f'{len(df_persist)}', '',
    'anhaltende Aktivitätsphasen')

df_results = pd.DataFrame(results)

# Schön anzeigen
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 50)
print('ERGEBNIS-TABELLE')
print('='*100)
display(df_results)

# Als CSV speichern
df_results.to_csv(f'{OUTPUT_DIR}/ergebnisse.csv', index=False, encoding='utf-8-sig')
print(f'\nGespeichert: {OUTPUT_DIR}/ergebnisse.csv')

## 15. PDF-Report erstellen

Alle Erkenntnisse, Grafiken und die Ergebnis-Tabelle werden in einem PDF zusammengefasst –  
ideal als Anhang für die Studienarbeit.

In [ ]:
# ── PDF-Report generieren ────────────────────────────────────────────
# Nutzt matplotlib's PdfPages – keine zusätzliche Bibliothek nötig.

from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from datetime import datetime as _dt
import textwrap

# Der Report liegt bewusst im selben Verzeichnis wie das Notebook (nicht in
# OUTPUT_DIR) - er ist das zentrale Abgabe-Dokument, kein Zwischenergebnis.
PDF_PATH = 'EEG_Report_BrewVis.pdf'

def text_page(pdf, title, lines, fontsize=10, title_size=16):
    """Erstellt eine Textseite im PDF."""
    fig = plt.figure(figsize=(8.27, 11.69))  # A4
    fig.text(0.08, 0.94, title, fontsize=title_size, fontweight='bold', color='#E85D04')
    fig.text(0.08, 0.915, '─' * 70, fontsize=8, color='#E85D04')
    y = 0.88
    for line in lines:
        if line.startswith('##'):
            y -= 0.012
            fig.text(0.08, y, line.lstrip('# '), fontsize=12, fontweight='bold', color='#1a1a2e')
            y -= 0.022
        elif line.startswith('#'):
            fig.text(0.08, y, line.lstrip('# '), fontsize=11, fontweight='bold')
            y -= 0.020
        else:
            fig.text(0.10, y, line, fontsize=fontsize, family='monospace' if line.startswith('  ') else 'sans-serif')
            y -= 0.0165
        if y < 0.06:
            break
    plt.axis('off')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

def image_page(pdf, image_path, title=None):
    """Fügt ein gespeichertes Bild als PDF-Seite ein."""
    if not os.path.exists(image_path):
        return
    fig = plt.figure(figsize=(11.69, 8.27))  # A4 quer
    if title:
        fig.text(0.5, 0.97, title, fontsize=13, fontweight='bold', ha='center', color='#1a1a2e')
    img = plt.imread(image_path)
    ax = fig.add_axes([0.04, 0.03, 0.92, 0.9])
    ax.imshow(img)
    ax.axis('off')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

def table_page(pdf, df, title, rows_per_page=24):
    """Rendert einen DataFrame als Tabelle im Querformat (mehrseitig bei Bedarf)."""
    # Interpretation-Spalte umbrechen, damit nichts abgeschnitten wird
    df_disp = df.copy()
    df_disp['Interpretation'] = df_disp['Interpretation'].apply(
        lambda s: '\n'.join(textwrap.wrap(str(s), width=42)))
    n_pages = int(np.ceil(len(df_disp) / rows_per_page))
    for pg in range(n_pages):
        chunk = df_disp.iloc[pg*rows_per_page:(pg+1)*rows_per_page]
        fig = plt.figure(figsize=(11.69, 8.27))  # A4 QUER
        ttl = title + (f' (Teil {pg+1}/{n_pages})' if n_pages > 1 else '')
        fig.text(0.5, 0.96, ttl, fontsize=13, fontweight='bold', ha='center', color='#E85D04')
        ax = fig.add_axes([0.03, 0.02, 0.94, 0.89])
        ax.axis('off')
        tbl = ax.table(cellText=chunk.values, colLabels=chunk.columns,
                       loc='center', cellLoc='left',
                       colWidths=[0.16, 0.20, 0.09, 0.07, 0.48])
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(7.5)
        tbl.scale(1, 1.5)
        for (r, c), cell in tbl.get_celld().items():
            cell.set_edgecolor('#ddd')
            if r == 0:
                cell.set_facecolor('#E85D04')
                cell.set_text_props(color='white', fontweight='bold')
        pdf.savefig(fig, bbox_inches='tight')
        plt.close()


print('Erstelle PDF-Report...')
with PdfPages(PDF_PATH) as pdf:
    # ── Titelseite ────────────────────────────────────────────────
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.text(0.5, 0.72, 'EEG-Auswertung', fontsize=30, fontweight='bold',
             ha='center', color='#E85D04')
    fig.text(0.5, 0.66, 'BrewVis – Informationsvisualisierung', fontsize=16, ha='center')
    fig.text(0.5, 0.62, 'Studienarbeit | OTH Amberg-Weiden', fontsize=12, ha='center', color='#666')
    fig.text(0.5, 0.50, f'Engagement-Index (β/[α+θ]): {mean_eng:.2f}  |  β/α-Verhältnis: {ba_ratio:.2f}',
             fontsize=12, ha='center', color='#444')
    fig.text(0.5, 0.44, f'{len(df_anom)} auffällige Momente  ·  {len(df_persist)} anhaltende Phasen',
             fontsize=12, ha='center', color='#444')
    fig.text(0.5, 0.08, f'Erstellt: {_dt.now().strftime("%d.%m.%Y %H:%M")}',
             fontsize=9, ha='center', color='#999')
    fig.text(0.5, 0.05, f'Aufnahme: {EEG_T0.strftime("%d.%m.%Y %H:%M")} | Dauer: {EEG_DURATION_S/60:.1f} min',
             fontsize=9, ha='center', color='#999')
    plt.axis('off')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

    # ── Zusammenfassungs-Seite (Klartext) ─────────────────────────
    n_theta_bursts = int((df_anom['type'] == 'Theta-Burst').sum()) if len(df_anom) else 0
    aktivierungs_text = {
        'HOCH':    'hohe Aktivierung, Person ist geistig dabei',
        'MITTEL':  'ausgewogene Aktivierung, Balance Fokus/Entspannung',
        'NIEDRIG': 'eher passiver/entspannter Zustand',
    }[level]

    summary_lines = [
        '## Worum geht es?',
        'Wir haben die Gehirnaktivitaet (EEG) einer Person gemessen, waehrend',
        'sie ein Video des Brauprozesses und dessen Visualisierung betrachtet hat.',
        'Ziel: Herausfinden, wie die Visualisierung auf die Aufmerksamkeit wirkt.',
        '',
        '## Die wichtigsten Ergebnisse',
        '',
        f'Aktivierung (Beta/Alpha): {ba_ratio:.2f} ({level})',
        f'  {aktivierungs_text}',
        '',
        f'Aufmerksamkeits-Trend: {trend_txt}',
        '',
        f'Hohe Aufmerksamkeit: {high_pct:.0f}% der Zeit',
        f'Niedrige Aufmerksamkeit: {low_pct:.0f}% der Zeit',
        '',
        f'Auffaellige Momente erkannt: {len(df_anom)}',
        f'Anhaltende Aktivitaetsphasen: {len(df_persist)}',
        '',
        '## Was bedeutet das fuer das Design?',
        '',
    ]
    # Design-Empfehlungen einfügen (aus den EEG-Daten abgeleitet, siehe Abschnitt 12)
    rec_lines = []
    rec_lines.append(f'1. Aktivierung von {ba_ratio:.2f} zeigt: Visualisierung bindet ein.')
    if ac > 15:
        rec_lines.append(f'2. Alpha steigt im Video (+{ac:.0f}%) - mehr Interaktion einbauen.')
    elif ac < -10:
        rec_lines.append(f'2. Alpha sinkt im Video (-{abs(ac):.0f}%) - gute Aufmerksamkeit.')
    if len(df_persist) > 0:
        tp = df_persist.iloc[0]
        rec_lines.append(f'3. Phase bei {tp["t_start"]:.0f}s pruefen (anhaltende Aktivitaet).')
    if n_theta_bursts > 2:
        rec_lines.append(f'4. {n_theta_bursts} Ueberlastungsmomente - Komplexitaet reduzieren.')
    rec_lines.append(f'{len(rec_lines)+1}. Okzipitales Delta hoch ({bp_o["Delta"]*100:.0f}%): Kontrast pruefen.')
    summary_lines.extend(rec_lines)
    
    text_page(pdf, 'Zusammenfassung (in einfachen Worten)', summary_lines)

    # ── Cockpit-Seite (Dashboard direkt nach der Zusammenfassung) ──
    # Frontal/Okzipital-Bandverteilung + PSD + Epochenvergleich auf einen Blick,
    # bevor es in die Detailanalysen geht.
    image_page(pdf, f'{OUTPUT_DIR}/07_dashboard.png', 'EEG-Cockpit: Gesamt-Dashboard')

    # ── Ergebnis-Tabelle ──────────────────────────────────────────
    table_page(pdf, df_results, 'Ergebnis-Tabelle: Alle Kennzahlen')

    # ── Grafik-Seiten (Detailanalysen) ─────────────────────────────
    graphics = [
        (f'{OUTPUT_DIR}/08_aufmerksamkeit_timeline.png','Aufmerksamkeits-Verlauf'),
        (f'{OUTPUT_DIR}/09_viertel_trend.png',          'Aufmerksamkeits-Trend (Ermuedung?)'),
        (f'{OUTPUT_DIR}/11_baseline_vs_video.png',      'Baseline vs. Video (Statistik)'),
        (f'{OUTPUT_DIR}/10_korrelation.png',            'Sehen vs. Denken (Kopplung)'),
        (f'{OUTPUT_DIR}/03_psd.png',                    'Frequenzspektrum'),
        (f'{OUTPUT_DIR}/06_anomalien.png',              'Auffaelligkeiten im Zeitverlauf'),
        (f'{OUTPUT_DIR}/05_spektrogramm.png',           'Spektrogramm'),
    ]
    for img_path, title in graphics:
        image_page(pdf, img_path, title)

    # ── Auffälligkeits-Screenshots ────────────────────────────────
    for i in range(len(df_anom)):
        panel = f'{OUTPUT_DIR}/screenshots/panel_{i+1:02d}.png'
        if os.path.exists(panel):
            image_page(pdf, panel, f'Auffaelligkeit #{i+1}')

    # ── Glossar-Seite ─────────────────────────────────────────────
    glossary_lines = [
        '## Frequenzbaender (Gehirnwellen)',
        'Delta (0.5-4 Hz):  Tiefschlaf / Artefakte',
        'Theta (4-8 Hz):    Konzentration ODER Muedigkeit',
        'Alpha (8-13 Hz):   Entspannung, Augen zu, Abschweifen',
        'Beta (13-30 Hz):   Aktives Denken, Fokus',
        'Gamma (30-45 Hz):  Intensive Verarbeitung',
        '',
        '## Kennzahlen',
        'Engagement-Index:  Wie aufmerksam/eingebunden (hoch=fokussiert)',
        'Beta/Alpha:        Aktivierung (>1.5 aktiv, <0.8 passiv)',
        'PSD:               Energie pro Frequenz (wie Musik-Equalizer)',
        '',
        '## Messpunkte',
        'Frontal:    Stirn - Konzentration, Aufmerksamkeit',
        'Okzipital:  Hinterkopf - visuelle Verarbeitung (Sehen)',
        '',
        '## Wichtiger Hinweis',
        'Diese Auswertung basiert auf EINER Person mit ZWEI Elektroden.',
        'Echte Studien nutzen 32-64 Elektroden und viele Personen.',
        'Die Ergebnisse sind daher explorativ (zeigen Tendenzen),',
        'aber gut geeignet fuer ein Studienprojekt.',
    ]
    text_page(pdf, 'Glossar: EEG-Begriffe erklaert', glossary_lines)

print(f'✓ PDF-Report erstellt: {PDF_PATH}')
print(f'  Groesse: {os.path.getsize(PDF_PATH)/1024:.0f} KB')

---
*EEG-Analyse – BrewVis InfoVis Studienarbeit | OTH Amberg-Weiden | SS 2026*